# Esqueletonizacion por Fragmento de Watershed (v1)

**Decision de diseno: nos saltamos el enlazado entre slices.** Para el bundle-scoring
punto-a-punto (distancia y orientacion entre vecinos), no necesitamos que un filamento
mantenga el mismo ID de principio a fin -- un filamento partido en varios fragmentos por
watershed sigue dando fragmentos validos: cada uno tiene su propia orientacion local correcta,
y dos fragmentos que son en realidad el mismo filamento real simplemente se cuentan como
"vecinos paralelos" entre si (igual de valido para la clasificacion bundle/mesh). El enlazado
solo aportaria continuidad de ID, que no es necesaria para esta metrica.

**Por que ahora SI tiene sentido esqueletonizar (incluso fragmentado):** al principio,
esqueletonizar el volumen binario completo alucinaba porque todos los filamentos compartian
voxels (un solo blob). Ahora, gracias a watershed + limpieza manual, cada fragmento (aunque
corto) tiene su propio ID y es un volumen 3D aislado de sus vecinos. Esqueletonizar CADA
fragmento por separado da una linea central limpia, sin riesgo de saltar a un vecino.

**Metodo:** `skimage.morphology.skeletonize` (en skimage >=0.19 detecta automaticamente 3D si
le pasas un volumen 3D).

**Input:** `labels_linked` -- el resultado de WATERSHED (`labels_ws_volume`), SIN enlazar entre
slices. El nombre de variable se mantiene por compatibilidad con el resto del notebook.

## 1. Carga

In [1]:
import numpy as np
from pathlib import Path
from scipy.ndimage import find_objects
from skimage.morphology import skeletonize
import mrcfile
import napari


output_dir = Path(r"C:\PhD\Actin_ET\Bundle Analysis\Amoeboid\L8_P14")  
VOXEL_SIZE_A = 12.4

# IMPORTANTE: saltamos el enlazado entre slices. Trabajamos directo sobre el resultado de
# WATERSHED (labels separados lateralmente pero NO enlazados/continuos entre Z). Esto esta bien
# para el bundle-scoring punto-a-punto: un filamento roto en varios fragmentos por watershed
# sigue dando fragmentos validos con orientacion local correcta -- no necesitamos que el ID
# se mantenga constante de principio a fin del filamento.
#
# La variable se sigue llamando 'labels_linked' por compatibilidad con el resto del notebook,
# pero el contenido real es el watershed SIN enlazar.
ws_input_path = output_dir / "C:\PhD\Actin_ET\Bundle Analysis\Amoeboid\L8_P14\L8_Position_14_actin_watershed_md4_ss2.5_er1.mrc"
if "labels_linked" not in dir():
    with mrcfile.open(str(ws_input_path), permissive=True) as mrc:
        labels_linked = mrc.data.copy().astype(np.int32)
    print(f"Cargado desde disco (watershed, SIN enlazar): {ws_input_path}")
else:
    print("Usando 'labels_linked' ya en memoria (no se toco el archivo en disco).")

print(f"Shape (Z,Y,X): {labels_linked.shape}")
n_filaments = labels_linked.max()
print(f"Numero de fragmentos de watershed (IDs, pueden ser fragmentos cortos): {n_filaments}")

Cargado desde disco (watershed, SIN enlazar): C:\PhD\Actin_ET\Bundle Analysis\Amoeboid\L8_P14\L8_Position_14_actin_watershed_md4_ss2.5_er1.mrc
Shape (Z,Y,X): (300, 1024, 1024)
Numero de fragmentos de watershed (IDs, pueden ser fragmentos cortos): 238


## 1b. Relabel: separar fragmentos lejanos que comparten el mismo ID

**El problema real detras de la lentitud:** sin enlazar entre slices, watershed REUTILIZA
numeros de ID en slices distintos -- el ID "106" en Z=10 puede ser un filamento en una esquina,
y el ID "106" en Z=150 puede ser un filamento totalmente distinto en la otra punta del volumen.
`find_objects` calcula el bounding box GLOBAL de cada numero de ID, que entonces abarca casi
todo el volumen (decenas de millones de voxels) aunque los voxels reales sean solo un par de
miles, concentrados en 2 puntos lejanos. Esqueletonizar ese bounding box gigante es lo que tarda
tanto.

**Solucion:** aplicamos `scipy.ndimage.label` (connected components 3D) RESTRINGIDO a la mascara
de cada ID de watershed, uno por uno. Esto separa automaticamente cualquier numero de ID que
en realidad correspondia a 2+ fragmentos lejanos (no conectados en 3D), dandole a cada
fragmento REAL un ID nuevo y unico. El bounding box de cada ID nuevo sera del tamano real del
fragmento, no del volumen completo.

In [2]:
from scipy.ndimage import label as cc_label

structure_26 = np.ones((3, 3, 3), dtype=bool)  # conectividad 26 para 3D

# OPTIMIZACION (importante para volumenes grandes): el metodo original hacia cc_label() para
# CADA id original (loop de cientos/miles de iteraciones), cada una recorriendo el volumen
# COMPLETO -- esto es lo que tardaba minutos/horas. La version de abajo hace cc_label() UNA
# SOLA VEZ sobre el binario completo (rapido), y solo despues revisa si algun componente
# geometrico resultante mezcla 2+ ids originales distintos (caso raro, tipicamente <1% del
# total) -- solo esos pocos casos se resuelven con un cc_label local, sobre su propio recorte
# pequeno (no sobre el volumen completo). Verificado con datos sinteticos que produce
# EXACTAMENTE la misma particion de voxels que el metodo original (mismos fragmentos, solo
# cambia la numeracion arbitraria de los IDs).

original_ids = np.unique(labels_linked[labels_linked > 0])
print(f"IDs originales de watershed: {len(original_ids)}")

# Paso 1: componentes geometricos puros (UNA pasada sobre el volumen completo)
binary_mask = labels_linked > 0
geom_components, n_geom = cc_label(binary_mask, structure=structure_26)
print(f"Componentes geometricos (conectividad pura, ignorando id original): {n_geom}")

# Paso 2: detectar que componentes geometricos mezclan 2+ ids originales distintos
geom_ids_flat = geom_components[binary_mask]
orig_ids_flat = labels_linked[binary_mask]
max_orig = int(labels_linked.max())
combo = geom_ids_flat.astype(np.int64) * (max_orig + 1) + orig_ids_flat.astype(np.int64)
unique_combos = np.unique(combo)
geom_id_per_combo = unique_combos // (max_orig + 1)
n_distinct_ids_per_geom = np.bincount(geom_id_per_combo, minlength=n_geom + 1)
problematic_geom_ids = sorted(np.where(n_distinct_ids_per_geom > 1)[0].tolist())
if 0 in problematic_geom_ids:
    problematic_geom_ids.remove(0)  # 0 = fondo, no es un componente real
print(f"Componentes que mezclan 2+ ids distintos (requieren resolucion local): "
      f"{len(problematic_geom_ids)} / {n_geom}")

# Paso 3: resolver SOLO los casos problematicos, usando coordenadas ya calculadas (sin
# volver a escanear el volumen completo por cada caso)
next_id = n_geom + 1
if problematic_geom_ids:
    problematic_array = np.array(problematic_geom_ids)
    is_problematic_in_mask = np.isin(geom_ids_flat, problematic_array)
    zs_all, ys_all, xs_all = np.nonzero(binary_mask)
    zs_prob = zs_all[is_problematic_in_mask]
    ys_prob = ys_all[is_problematic_in_mask]
    xs_prob = xs_all[is_problematic_in_mask]
    geom_id_prob = geom_ids_flat[is_problematic_in_mask]

    for geom_id in problematic_geom_ids:
        sel = (geom_id_prob == geom_id)
        zs_c, ys_c, xs_c = zs_prob[sel], ys_prob[sel], xs_prob[sel]
        z0b, z1b = int(zs_c.min()), int(zs_c.max()) + 1
        y0b, y1b = int(ys_c.min()), int(ys_c.max()) + 1
        x0b, x1b = int(xs_c.min()), int(xs_c.max()) + 1

        crop_geom = geom_components[z0b:z1b, y0b:y1b, x0b:x1b]
        crop_orig = labels_linked[z0b:z1b, y0b:y1b, x0b:x1b]
        mask_this_geom = (crop_geom == geom_id)

        sub_orig_ids = np.unique(crop_orig[mask_this_geom])
        for sub_id in sub_orig_ids:
            sub_mask = mask_this_geom & (crop_orig == sub_id)
            sub_components, n_sub = cc_label(sub_mask, structure=structure_26)
            for c in range(1, n_sub + 1):
                view = geom_components[z0b:z1b, y0b:y1b, x0b:x1b]
                view[sub_components == c] = next_id
                geom_components[z0b:z1b, y0b:y1b, x0b:x1b] = view
                next_id += 1

labels_relabeled = geom_components  # ya es el resultado final, sin copia extra

print(f"IDs despues de separar fragmentos lejanos: {next_id - 1}")
print(f"(Si este numero es MUCHO mayor que el original, confirma que muchos IDs de watershed")
print(f" estaban reciclados entre fragmentos lejanos no conectados.)")

# A partir de aqui, el resto del notebook usa 'labels_relabeled' en vez de 'labels_linked'
labels_linked = labels_relabeled
n_filaments = labels_linked.max()

IDs originales de watershed: 238
Componentes geometricos (conectividad pura, ignorando id original): 119
Componentes que mezclan 2+ ids distintos (requieren resolucion local): 117 / 119
IDs despues de separar fragmentos lejanos: 17541
(Si este numero es MUCHO mayor que el original, confirma que muchos IDs de watershed
 estaban reciclados entre fragmentos lejanos no conectados.)


## 1c. Diagnostico de tamano de bounding box (antes de esqueletonizar)

`skeletonize` en 3D escala con el VOLUMEN del recorte (bounding box), no solo con los voxels
reales del filamento. Si algun filamento tiene un bounding box anormalmente grande (por ejemplo,
si el enlazado fusiono por error 2+ filamentos en un solo ID, o si algun ID quedo con un blob
grande sin limpiar del todo), ese filamento por si solo puede tardar mucho y bloquear todo el
loop. Revisamos esto ANTES de lanzar la esqueletonizacion completa.

In [3]:
objects = find_objects(labels_linked)

bbox_volumes = []
bbox_info = []

for fid_minus_1, obj in enumerate(objects):
    if obj is None:
        continue
    fid = fid_minus_1 + 1
    z_slice, y_slice, x_slice = obj
    dz = z_slice.stop - z_slice.start
    dy = y_slice.stop - y_slice.start
    dx = x_slice.stop - x_slice.start
    vol = dz * dy * dx
    bbox_volumes.append(vol)
    bbox_info.append((fid, dz, dy, dx, vol))

bbox_volumes = np.array(bbox_volumes)
print(f"Volumen de bounding box (voxels) por filamento: "
      f"min={bbox_volumes.min():,}, median={np.median(bbox_volumes):.0f}, max={bbox_volumes.max():,}")

# Mostrar los 10 filamentos con bounding box mas grande -- candidatos a cuello de botella
bbox_info.sort(key=lambda x: -x[4])
print(f"\nTop 10 filamentos por tamano de bounding box (fid, dz, dy, dx, volumen):")
for fid, dz, dy, dx, vol in bbox_info[:10]:
    print(f"  ID={fid}: dz={dz}, dy={dy}, dx={dx}, volumen={vol:,} voxels")

print(f"\n-> Si el mas grande es ordenes de magnitud mayor que la mediana, revisalo en Napari")
print(f"   antes de esqueletonizar -- puede ser un filamento mal enlazado/fusionado.")

Volumen de bounding box (voxels) por filamento: min=1, median=57, max=17,226

Top 10 filamentos por tamano de bounding box (fid, dz, dy, dx, volumen):
  ID=14845: dz=11, dy=29, dx=54, volumen=17,226 voxels
  ID=17525: dz=11, dy=14, dx=72, volumen=11,088 voxels
  ID=14843: dz=6, dy=31, dx=47, volumen=8,742 voxels
  ID=17400: dz=7, dy=20, dx=56, volumen=7,840 voxels
  ID=644: dz=14, dy=10, dx=51, volumen=7,140 voxels
  ID=17460: dz=15, dy=15, dx=30, volumen=6,750 voxels
  ID=17512: dz=11, dy=12, dx=46, volumen=6,072 voxels
  ID=14825: dz=8, dy=22, dx=33, volumen=5,808 voxels
  ID=14828: dz=8, dy=20, dx=35, volumen=5,600 voxels
  ID=14849: dz=4, dy=18, dx=75, volumen=5,400 voxels

-> Si el mas grande es ordenes de magnitud mayor que la mediana, revisalo en Napari
   antes de esqueletonizar -- puede ser un filamento mal enlazado/fusionado.


## 2. Esqueletonizar cada filamento por separado

Para cada ID persistente:
1. Se recorta su bounding box (via `find_objects`, rapido) con un pequeno padding.
2. Se esqueletoniza SOLO esa region recortada (mucho mas rapido que esqueletonizar el volumen
   completo repetidas veces, y evita cualquier interaccion con voxels de otros filamentos).
3. El resultado (voxels del esqueleto) se escribe de vuelta en un volumen completo, manteniendo
   el mismo ID persistente.

Esto procesa cada filamento de forma independiente -- ningun filamento puede "robarle" voxels
del esqueleto a otro, porque cada uno se esqueletoniza aislado en su propio recorte.

In [4]:
PADDING = 2  # voxels de margen alrededor del bounding box de cada filamento

# 'objects' ya se calculo en la celda de diagnostico (1b) -- lo reutilizamos
skeleton_volume = np.zeros_like(labels_linked, dtype=np.int32)

n_skeletonized = 0
n_empty_skeleton = 0  # filamentos cuyo esqueleto salio vacio (demasiado pequenos/delgados)

for fid_minus_1, obj in enumerate(objects):
    if obj is None:
        continue
    fid = fid_minus_1 + 1

    # Bounding box con padding, recortado a los limites del volumen
    z_slice, y_slice, x_slice = obj
    z0 = max(z_slice.start - PADDING, 0)
    z1 = min(z_slice.stop + PADDING, labels_linked.shape[0])
    y0 = max(y_slice.start - PADDING, 0)
    y1 = min(y_slice.stop + PADDING, labels_linked.shape[1])
    x0 = max(x_slice.start - PADDING, 0)
    x1 = min(x_slice.stop + PADDING, labels_linked.shape[2])

    crop = labels_linked[z0:z1, y0:y1, x0:x1]
    mask_this_filament = (crop == fid)

    skeleton_crop = skeletonize(mask_this_filament)

    if skeleton_crop.sum() == 0:
        n_empty_skeleton += 1
        continue

    # Escribir de vuelta en el volumen completo, en la posicion original
    skeleton_volume[z0:z1, y0:y1, x0:x1][skeleton_crop] = fid
    n_skeletonized += 1

print(f"Filamentos esqueletonizados: {n_skeletonized} / {n_filaments}")
print(f"Filamentos con esqueleto vacio (demasiado pequenos/delgados): {n_empty_skeleton}")

Filamentos esqueletonizados: 17394 / 17541
Filamentos con esqueleto vacio (demasiado pequenos/delgados): 30


## 3. Sanity checks

- **Voxels del esqueleto por filamento:** un esqueleto de un filamento recto deberia tener un
  numero de voxels del orden de su longitud en Z (una linea de 1 voxel de espesor), NO del orden
  de su volumen original (que incluye el grosor de ~5-6 voxels de diametro).
- **Ratio esqueleto/extension-Z:** cerca de 1.0 indica una linea limpia sin ramas. `skeletonize`
  ya poda por si solo las protuberancias muy cortas (ruido de borde de 1-4 voxels, verificado con
  un test sintetico), asi que un ratio bajo no significa ausencia total de irregularidades en el
  borde original -- solo que no sobrevivieron como rama en el esqueleto. Un ratio notablemente
  > 1.3-1.5 SI sugiere una rama larga real que merece revisarse (puede ser ruido significativo,
  o una curvatura/bifurcacion genuina del filamento).

In [5]:
# OPTIMIZACION: el metodo original hacia (labels_linked == fid).sum() POR CADA fragmento --
# cada comparacion recorre el volumen completo, asi que con miles de fragmentos esto escala
# muy mal (mismo patron de lentitud que ya resolvimos en el relabeling). np.bincount() cuenta
# voxels por cada valor entero en UNA SOLA pasada sobre el array, sin importar cuantos
# fragmentos distintos haya -- mucho mas rapido y escala perfectamente con miles de IDs.

n_max_id = int(labels_linked.max())

orig_voxel_counts_by_id = np.bincount(labels_linked.ravel(), minlength=n_max_id + 1)
skel_voxel_counts_by_id = np.bincount(skeleton_volume.ravel(), minlength=n_max_id + 1)

# z_extent por fragmento: seguimos usando 'objects' (de find_objects, ya calculado antes,
# barato) para el rango Z de cada uno
z_extents_by_id = np.zeros(n_max_id + 1, dtype=int)
for fid_minus_1, obj in enumerate(objects):
    if obj is None:
        continue
    fid = fid_minus_1 + 1
    if fid <= n_max_id:
        z_extents_by_id[fid] = obj[0].stop - obj[0].start

# Quedarnos solo con los fragmentos que existen en AMBOS volumenes (original y esqueleto)
valid_ids = np.where((orig_voxel_counts_by_id[1:] > 0) & (skel_voxel_counts_by_id[1:] > 0))[0] + 1

orig_voxel_counts = orig_voxel_counts_by_id[valid_ids]
skel_voxel_counts = skel_voxel_counts_by_id[valid_ids]
z_extents = z_extents_by_id[valid_ids]

print(f"Voxels originales por filamento: median={np.median(orig_voxel_counts):.0f}")
print(f"Voxels de esqueleto por filamento: median={np.median(skel_voxel_counts):.0f}")
print(f"Extension en Z por filamento: median={np.median(z_extents):.0f}")
print(f"\nRatio esqueleto/extension-Z (deberia estar cerca de 1.0 para un filamento recto sin ramas):")
ratio = skel_voxel_counts / np.maximum(z_extents, 1)
print(f"  median={np.median(ratio):.2f}, max={ratio.max():.2f}")
print(f"  Filamentos con ratio > 1.5 (posibles ramas espurias): {(ratio > 1.5).sum()} "
      f"({100*(ratio>1.5).sum()/len(ratio):.1f}%)")

Voxels originales por filamento: median=43
Voxels de esqueleto por filamento: median=12
Extension en Z por filamento: median=1

Ratio esqueleto/extension-Z (deberia estar cerca de 1.0 para un filamento recto sin ramas):
  median=12.00, max=144.00
  Filamentos con ratio > 1.5 (posibles ramas espurias): 17194 (98.9%)


## 4. Visualizar en Napari (3D)

In [6]:
viewer_skel = napari.Viewer(title="Esqueletos por filamento (3D)")
viewer_skel.add_labels(labels_linked, name="filamentos (volumen)", opacity=0.25)
viewer_skel.add_labels(skeleton_volume, name="esqueletos", opacity=1.0)
viewer_skel.dims.ndisplay = 3

print("Cada esqueleto deberia verse como una linea fina y limpia dentro de su filamento.")
print("Si ves 'pelos' o ramas cortas saliendo del eje principal, es ruido de borde --")
print("se puede limpiar despues con un filtro de poda (pruning) si es necesario.")

Cada esqueleto deberia verse como una linea fina y limpia dentro de su filamento.
Si ves 'pelos' o ramas cortas saliendo del eje principal, es ruido de borde --
se puede limpiar despues con un filtro de poda (pruning) si es necesario.


## 5. Guardar resultado

In [7]:
skel_output_path = output_dir / (ws_input_path.stem + "_skeleton.mrc")

max_label = skeleton_volume.max()
if max_label > 32767:
    raise ValueError(f"max_label={max_label} excede el rango de int16. Avisa si esto pasa.")

with mrcfile.new(str(skel_output_path), overwrite=True) as mrc_out:
    mrc_out.set_data(skeleton_volume.astype(np.int16))
    mrc_out.voxel_size = VOXEL_SIZE_A

print(f"Guardado: {skel_output_path}")

Guardado: C:\PhD\Actin_ET\Bundle Analysis\Amoeboid\L8_P14\L8_Position_14_actin_watershed_md4_ss2.5_er1_skeleton.mrc


## Proximos pasos

Con un esqueleto limpio por filamento (1 voxel de espesor, ID persistente), el siguiente paso
es extraer las coordenadas 3D del esqueleto de cada filamento, ordenarlas a lo largo de su eje
principal (Y), y usarlas para calcular orientacion local (vectores tangentes) y distancias
inter-filamento -- el objetivo final del bundle-scoring.